# Module 04: Stability Selection

## Learning Objectives

By completing this notebook, you will:
1. Implement stability selection from scratch using Lasso as the base estimator
2. Visualise stability paths — selection probability vs. lambda — for all features
3. Compare stable features to single-Lasso-selected features on a correlated dataset
4. Demonstrate the robustness advantage of stability selection across bootstrap samples
5. Apply Group Lasso to a dataset with known group structure

## Prerequisites
- Guide 02: Stability Selection and Structured Sparsity
- Notebook 01: Regularisation Paths

## Estimated Time: 15 minutes

---

## 1. Setup and Data

We use a controlled synthetic dataset where we know the ground truth: which features are truly relevant. This lets us measure false positive and false negative rates precisely.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.linear_model import Lasso, LassoCV, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

np.random.seed(0)
plt.style.use('seaborn-v0_8-whitegrid')

print("Imports complete.")

In [ ]:
# Construct a controlled dataset with known ground truth
# True support: features 0, 1, 2, 3, 4 (first 5 features)
# Features 0–2 form a correlated group (r~0.9)
# Features 3–4 are independent true predictors
# Features 5–19 are null (independent noise)

def make_correlated_dataset(n=400, p=20, n_true=5, corr_strength=0.9, noise_std=1.0, seed=0):
    """
    Create a regression dataset with known ground truth.

    Parameters
    ----------
    n : int — number of observations
    p : int — total features
    n_true : int — number of truly relevant features
    corr_strength : float — correlation within the first group
    noise_std : float — standard deviation of outcome noise

    Returns
    -------
    X : pd.DataFrame (n, p) — standardised features
    y : np.ndarray (n,) — outcome
    true_features : list — indices of truly relevant features
    true_coef : np.ndarray (p,) — true coefficients
    """
    rng = np.random.default_rng(seed)
    
    # Correlated group: features 0, 1, 2
    z_group = rng.normal(0, 1, n)  # shared latent factor
    X_corr = np.column_stack([
        corr_strength * z_group + np.sqrt(1 - corr_strength**2) * rng.normal(0, 1, n),
        corr_strength * z_group + np.sqrt(1 - corr_strength**2) * rng.normal(0, 1, n),
        corr_strength * z_group + np.sqrt(1 - corr_strength**2) * rng.normal(0, 1, n),
    ])
    
    # Independent true features: 3, 4
    X_indep_true = rng.normal(0, 1, (n, 2))
    
    # Null features: 5–19
    X_null = rng.normal(0, 1, (n, p - n_true))
    
    X_raw = np.hstack([X_corr, X_indep_true, X_null])
    
    # True coefficients: 2.0 for all true features
    true_coef = np.zeros(p)
    true_coef[:n_true] = 2.0
    
    # Generate outcome
    y = X_raw @ true_coef + rng.normal(0, noise_std, n)
    
    # Standardise
    scaler = StandardScaler()
    X = pd.DataFrame(
        scaler.fit_transform(X_raw),
        columns=[f'x{i:02d}' for i in range(p)]
    )
    
    true_features = list(range(n_true))
    return X, y, true_features, true_coef


X, y, TRUE_FEATURES, true_coef = make_correlated_dataset(
    n=400, p=20, n_true=5, corr_strength=0.9, noise_std=1.0
)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"True features: {TRUE_FEATURES} ({X.columns[TRUE_FEATURES].tolist()})")
print(f"Null features: {list(range(5, 20))}")
print(f"\nCorrelation among first 3 features (should be ~0.9):")
print(X.iloc[:, :3].corr().round(3))

## 2. Stability Selection: From Scratch

The implementation has three parts:
1. For each subsample: compute the Lasso path
2. Record which features are selected at each $\lambda$ value
3. Average across subsamples to get selection probabilities

In [ ]:
def stability_selection(
    X, y, n_subsamples=100, subsample_fraction=0.5,
    n_alphas=50, eps=1e-2, random_state=0
):
    """
    Stability selection via repeated Lasso path fitting on random subsamples.

    Parameters
    ----------
    X : array (n, p) — standardised feature matrix
    y : array (n,) — target
    n_subsamples : int — number of bootstrap subsamples (>=100 recommended)
    subsample_fraction : float — fraction of data per subsample (0.5 recommended)
    n_alphas : int — number of lambda values in the path
    eps : float — lambda_min = eps * lambda_max
    random_state : int — reproducibility seed

    Returns
    -------
    selection_probs : array (p, n_alphas) — selection probability per feature per lambda
    alphas : array (n_alphas,) — lambda grid (decreasing)
    """
    rng = np.random.default_rng(random_state)
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)
    n, p = X_arr.shape
    subsample_size = int(np.floor(subsample_fraction * n))

    # Determine the alpha grid from the full dataset path
    # (ensures consistent grid across all subsamples)
    alphas_full, _, _ = lasso_path(X_arr, y_arr, n_alphas=n_alphas, eps=eps)
    alphas = alphas_full  # shape (n_alphas,)

    # Accumulate selection counts: shape (p, n_alphas)
    selection_counts = np.zeros((p, n_alphas), dtype=float)

    for b in range(n_subsamples):
        # Draw random subsample WITHOUT replacement
        idx = rng.choice(n, size=subsample_size, replace=False)
        X_sub = X_arr[idx]
        y_sub = y_arr[idx]

        # Fit Lasso path on this subsample, using the same alpha grid
        # Note: alphas passed explicitly ensures the same grid for all subsamples
        try:
            _, coefs_sub, _ = lasso_path(
                X_sub, y_sub,
                alphas=alphas,  # fixed grid — enables averaging
                max_iter=500
            )
        except Exception:
            continue  # skip degenerate subsamples

        # coefs_sub shape: (p, n_alphas)
        # Count nonzero entries (selected features at each lambda)
        selection_counts += (coefs_sub != 0).astype(float)

    # Selection probability = fraction of subsamples where feature was selected
    selection_probs = selection_counts / n_subsamples

    return selection_probs, alphas


# Run stability selection (100 subsamples)
print("Running stability selection (100 subsamples × 50 lambda values)...")
sel_probs, alphas = stability_selection(
    X, y,
    n_subsamples=100,
    subsample_fraction=0.5,
    n_alphas=50,
    eps=1e-2,
    random_state=0
)
print(f"Selection probability matrix: {sel_probs.shape} (p={sel_probs.shape[0]}, lambdas={sel_probs.shape[1]})")
print("Done.")

## 3. Visualise the Stability Path

The stability path shows selection probability vs. $\log(\lambda)$ for each feature. True features should rise and plateau near 1.0; null features should stay near 0.

In [ ]:
# Stability path plot
fig, ax = plt.subplots(figsize=(11, 6))

feature_names_all = X.columns.tolist()

for i in range(X.shape[1]):
    is_true = i in TRUE_FEATURES
    color = '#e41a1c' if is_true else '#aaaaaa'
    lw = 2.5 if is_true else 1.0
    alpha_line = 1.0 if is_true else 0.5
    zorder = 3 if is_true else 1
    
    label = feature_names_all[i] if is_true else None
    ax.plot(
        np.log10(alphas), sel_probs[i],
        color=color, linewidth=lw, alpha=alpha_line,
        zorder=zorder, label=label
    )

# Threshold line
PI_THR = 0.75
ax.axhline(PI_THR, color='navy', linestyle='--', linewidth=2, label=f'Threshold π={PI_THR}')

ax.set_xlabel('log₁₀(λ)', fontsize=13)
ax.set_ylabel('Selection Probability', fontsize=13)
ax.set_title('Stability Path\nRed = true features, Grey = null features', fontsize=14)
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=10, loc='upper right')
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('stability_path.png', dpi=100, bbox_inches='tight')
plt.show()

# Maximum stability probability for each feature
max_probs = sel_probs.max(axis=1)
stable_features = np.where(max_probs >= PI_THR)[0]

print(f"\nFeatures with max probability >= {PI_THR}:")
for i in stable_features:
    is_true = '(TRUE)' if i in TRUE_FEATURES else '(NULL - FALSE POSITIVE)'
    print(f"  {feature_names_all[i]}: max_prob={max_probs[i]:.3f} {is_true}")

## 4. Stability Selection vs. Single Lasso: Side-by-Side Comparison

Compare the selection outputs of a single LassoCV fit against stability selection. The key question: how do they handle the correlated feature group `{x00, x01, x02}`?

In [ ]:
# Single LassoCV fit
lasso_cv = LassoCV(cv=5, n_alphas=50, max_iter=5000, random_state=0)
lasso_cv.fit(X.values, y)

lasso_selected = set(np.where(lasso_cv.coef_ != 0)[0])
stable_selected = set(stable_features.tolist())
true_set = set(TRUE_FEATURES)

def selection_metrics(selected_set, true_set, p):
    """Compute TP, FP, FN, precision, recall for a selection."""
    tp = len(selected_set & true_set)
    fp = len(selected_set - true_set)
    fn = len(true_set - selected_set)
    precision = tp / max(len(selected_set), 1)
    recall = tp / max(len(true_set), 1)
    return tp, fp, fn, precision, recall

tp_l, fp_l, fn_l, prec_l, rec_l = selection_metrics(lasso_selected, true_set, X.shape[1])
tp_s, fp_s, fn_s, prec_s, rec_s = selection_metrics(stable_selected, true_set, X.shape[1])

print("Comparison: LassoCV vs Stability Selection")
print("=" * 60)
print(f"{'Metric':<25} {'LassoCV':>15} {'Stability':>15}")
print("-" * 60)
print(f"{'Selected features':<25} {len(lasso_selected):>15} {len(stable_selected):>15}")
print(f"{'True positives':<25} {tp_l:>15} {tp_s:>15}")
print(f"{'False positives':<25} {fp_l:>15} {fp_s:>15}")
print(f"{'False negatives':<25} {fn_l:>15} {fn_s:>15}")
print(f"{'Precision':<25} {prec_l:>15.3f} {prec_s:>15.3f}")
print(f"{'Recall':<25} {rec_l:>15.3f} {rec_s:>15.3f}")

print(f"\nLassoCV selected from correlated group {{x00,x01,x02}}: ", end="")
corr_lasso = [f'x{i:02d}' for i in sorted(lasso_selected & {0, 1, 2})]
print(corr_lasso)

print(f"Stability selected from correlated group {{x00,x01,x02}}: ", end="")
corr_stable = [f'x{i:02d}' for i in sorted(stable_selected & {0, 1, 2})]
print(corr_stable)

In [ ]:
# Visual comparison: selection probability bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: LassoCV coefficients
ax = axes[0]
coef_abs = np.abs(lasso_cv.coef_)
colors_l = ['#e41a1c' if i in TRUE_FEATURES else '#aaaaaa' for i in range(X.shape[1])]
ax.bar(feature_names_all, coef_abs, color=colors_l)
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('|Coefficient|', fontsize=11)
ax.set_title('LassoCV Coefficients\n(Red = true features)', fontsize=12)
ax.tick_params(axis='x', rotation=90)

# Right: Stability selection probabilities
ax = axes[1]
colors_s = ['#e41a1c' if i in TRUE_FEATURES else '#aaaaaa' for i in range(X.shape[1])]
ax.bar(feature_names_all, max_probs, color=colors_s)
ax.axhline(PI_THR, color='navy', linestyle='--', linewidth=2, label=f'Threshold {PI_THR}')
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Max Selection Probability', fontsize=11)
ax.set_title('Stability Selection Probabilities\n(Red = true features)', fontsize=12)
ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=90)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('lasso_vs_stability_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Robustness Advantage: Stability Across Bootstrap Samples

To make the fragility of single Lasso fits concrete, we run LassoCV 30 times on different bootstrap subsamples and record which features are selected each time. High variance in selection = fragile. Low variance = robust.

In [ ]:
# Run LassoCV 30 times on different 50% subsamples
# For each run, record which features are selected
n_runs = 30
selection_matrix = np.zeros((n_runs, X.shape[1]), dtype=int)

for run in range(n_runs):
    rng_run = np.random.default_rng(run)
    idx = rng_run.choice(len(y), size=len(y) // 2, replace=False)
    X_sub = X.values[idx]
    y_sub = y[idx]
    
    lasso_run = LassoCV(cv=3, n_alphas=30, max_iter=2000, random_state=run)
    lasso_run.fit(X_sub, y_sub)
    selection_matrix[run] = (lasso_run.coef_ != 0).astype(int)

# Selection frequency across runs
selection_freq = selection_matrix.mean(axis=0)

# Plot: heatmap of selections across runs
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
ax = axes[0]
im = ax.imshow(selection_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(X.shape[1]))
ax.set_xticklabels(feature_names_all, rotation=90, fontsize=8)
ax.set_yticks(range(0, n_runs, 5))
ax.set_yticklabels([f'Run {i}' for i in range(0, n_runs, 5)], fontsize=9)
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Subsample Run', fontsize=11)
ax.set_title('LassoCV Selections Across 30 Subsamples\nGreen=selected, Red=not selected', fontsize=11)

# Add vertical lines separating true from null features
ax.axvline(4.5, color='blue', linewidth=2, label='True/Null boundary')
plt.colorbar(im, ax=ax)

# Frequency bar chart
ax = axes[1]
colors_freq = ['#e41a1c' if i in TRUE_FEATURES else '#aaaaaa' for i in range(X.shape[1])]
ax.bar(feature_names_all, selection_freq, color=colors_freq)
ax.axhline(0.5, color='navy', linestyle='--', linewidth=1.5, label='50% line')
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Selection Frequency (30 runs)', fontsize=11)
ax.set_title('Feature Selection Frequency\nAcross Subsamples', fontsize=12)
ax.tick_params(axis='x', rotation=90)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('lasso_robustness.png', dpi=100, bbox_inches='tight')
plt.show()

print("Selection variability (std across runs):")
sel_std = selection_matrix.std(axis=0)
for i, (name, freq, std) in enumerate(zip(feature_names_all, selection_freq, sel_std)):
    tag = '(TRUE)' if i in TRUE_FEATURES else '(null)'
    print(f"  {name} {tag}: freq={freq:.2f}, std={std:.2f}")

## 6. Group Lasso for Structured Feature Selection

Group Lasso selects all features within a group or none. Here we define groups matching the true data structure: `{x00, x01, x02}` as Group 0, `{x03, x04}` as Group 1, and `{x05, ..., x09}` as Group 2 (null). We implement block coordinate descent for Group Lasso from scratch.

In [ ]:
def group_lasso_bcd(
    X, y, groups, lam, max_iter=500, tol=1e-6
):
    """
    Solve Group Lasso by block coordinate descent.

    Parameters
    ----------
    X : array (n, p)
    y : array (n,)
    groups : list of lists — each inner list is feature indices for one group
    lam : float — regularisation parameter
    max_iter : int — maximum BCD iterations
    tol : float — convergence tolerance on max coefficient change

    Returns
    -------
    beta : array (p,) — Group Lasso coefficient vector
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    beta = np.zeros(p)
    
    for iteration in range(max_iter):
        beta_old = beta.copy()
        
        for g, group_idx in enumerate(groups):
            group_idx = list(group_idx)
            p_g = len(group_idx)
            
            # Compute partial residual: exclude current group from fitted values
            other_idx = [i for i in range(p) if i not in group_idx]
            r_g = y - X[:, other_idx] @ beta[other_idx]
            
            # Residual correlation for this group: S_g = X_g^T r_g
            X_g = X[:, group_idx]   # (n, p_g)
            S_g = X_g.T @ r_g      # (p_g,)
            
            # Group size normalisation: lambda * sqrt(p_g)
            # This ensures fair penalisation regardless of group size
            threshold = lam * np.sqrt(p_g)
            
            S_g_norm = np.linalg.norm(S_g)
            
            if S_g_norm <= threshold:
                # Group is inactive: set all to zero
                beta[group_idx] = 0.0
            else:
                # Block soft-thresholding update
                # This is the vector analogue of scalar soft-thresholding
                # Shrink S_g toward zero in L2 norm by fraction (1 - threshold/||S_g||)
                # Note: we need to account for the Gram matrix X_g^T X_g
                # Simplified update (assumes X is standardised, diagonal Gram approx.)
                scale = (1.0 - threshold / S_g_norm)
                # Proper update: beta_g = (X_g^T X_g)^{-1} S_g * scale
                # For standardised X: X_g^T X_g ≈ n * I, so divide by n
                beta[group_idx] = scale * S_g / n
        
        # Check convergence
        if np.max(np.abs(beta - beta_old)) < tol:
            break
    
    return beta


# Define groups matching data structure
groups = [
    [0, 1, 2],    # Group 0: correlated true features
    [3, 4],       # Group 1: independent true features
    [5, 6, 7, 8, 9],      # Group 2: null features
    [10, 11, 12, 13, 14], # Group 3: null features
    [15, 16, 17, 18, 19], # Group 4: null features
]

# Fit Group Lasso at a moderate lambda
lam_gl = 0.15
beta_gl = group_lasso_bcd(X.values, y, groups, lam=lam_gl)

print(f"Group Lasso (lambda={lam_gl}):")
for g, group_idx in enumerate(groups):
    group_coef = beta_gl[group_idx]
    group_norm = np.linalg.norm(group_coef)
    is_active = group_norm > 1e-8
    true_tag = 'TRUE' if all(i in TRUE_FEATURES for i in group_idx) else \
               'MIXED' if any(i in TRUE_FEATURES for i in group_idx) else 'NULL'
    print(f"  Group {g} {[X.columns[i] for i in group_idx]} ({true_tag}): "
          f"norm={group_norm:.4f}, {'ACTIVE' if is_active else 'ZERO'}")

In [ ]:
# Group Lasso path: vary lambda, track group norms
lambda_grid = np.geomspace(0.5, 0.01, 40)
group_norms = np.zeros((len(groups), len(lambda_grid)))

for j, lam in enumerate(lambda_grid):
    beta_j = group_lasso_bcd(X.values, y, groups, lam=lam, max_iter=300)
    for g, group_idx in enumerate(groups):
        group_norms[g, j] = np.linalg.norm(beta_j[group_idx])

group_colors = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']
group_labels = ['Group0 (true, corr)', 'Group1 (true, indep)', 'Group2 (null)', 'Group3 (null)', 'Group4 (null)']

fig, ax = plt.subplots(figsize=(10, 5))

for g in range(len(groups)):
    ls = '-' if g < 2 else '--'
    lw = 2.5 if g < 2 else 1.5
    ax.plot(np.log10(lambda_grid), group_norms[g], color=group_colors[g],
            linewidth=lw, linestyle=ls, label=group_labels[g])

ax.set_xlabel('log₁₀(λ)', fontsize=12)
ax.set_ylabel('Group L2 Norm', fontsize=12)
ax.set_title('Group Lasso Path — Group L2 Norms vs. λ', fontsize=14)
ax.legend(fontsize=10)
ax.invert_xaxis()

plt.tight_layout()
plt.savefig('group_lasso_path.png', dpi=100, bbox_inches='tight')
plt.show()

print("Observation: True groups enter the model as groups (norm goes from 0 to nonzero).")
print("Null groups remain at zero for higher lambda values than individual-feature Lasso.")

## Exercise 6.1: Threshold Selection Using the Theoretical Bound

**Task:** Using the Meinshausen-Bühlmann bound, compute the minimum threshold $\pi_\text{thr}$ required to guarantee at most 1 expected false positive.

Given:
- $p = 20$ features
- $q = 0.5$ (subsample fraction)
- $\mathbb{E}[|\hat{S}^\lambda|]$: use the average number of features selected across all subsamples at the lambda that maximises the total selection count

The bound is:

$$\mathbb{E}[\text{FP}] \leq \frac{1}{2\pi_{\text{thr}} - 1} \cdot \frac{q^2}{1-q} \cdot \frac{\mathbb{E}[|\hat{S}^\lambda|]^2}{p} \leq V$$

Solving for $\pi_\text{thr}$:

$$\pi_\text{thr} \geq \frac{1}{2}\left(1 + \frac{q^2}{1-q} \cdot \frac{\mathbb{E}[|\hat{S}^\lambda|]^2}{p \cdot V}\right)$$

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Compute the average model size at each lambda value
#    (average across subsamples — use sel_probs which has shape (p, n_alphas))
#    avg_model_size[j] = sum_i sel_probs[i, j] = average number of selected features at lambda j

# 2. Find the lambda that maximises the average model size
#    E_S_lambda = maximum of avg_model_size

# 3. Compute the minimum pi_thr for V=1 false positive, q=0.5, p=20
#    Use the formula above

# 4. Print E_S_lambda and pi_thr

In [ ]:
# AUTO-GRADED TESTS — do not modify
def test_exercise_61():
    p = 20
    q = 0.5
    V = 1
    
    avg_model_size = sel_probs.sum(axis=0)  # (n_alphas,)
    E_S_lambda = avg_model_size.max()
    
    pi_thr_min = 0.5 * (1 + (q**2 / (1 - q)) * (E_S_lambda**2 / (p * V)))
    
    print(f"Max average model size E[|S_lambda|]: {E_S_lambda:.2f}")
    print(f"Required threshold pi_thr >= {pi_thr_min:.4f} for <= {V} expected false positive")
    
    assert pi_thr_min > 0.5, "Threshold must be > 0.5 by construction"
    assert pi_thr_min <= 1.0, f"Threshold {pi_thr_min:.3f} > 1.0: model size too large for guarantee. Increase lambda."
    
    print(f"Interpretation: use threshold >= {pi_thr_min:.2f} to control to <= 1 expected false positive.")
    print("Test passed.")

test_exercise_61()

## Exercise 6.2: Vary the Threshold

**Task:** Run the stable feature selection at three different thresholds: $\pi_\text{thr} \in \{0.6, 0.75, 0.9\}$. For each, report:
- Number of stable features selected
- True positives, false positives
- Precision and recall

Observe the precision-recall tradeoff as the threshold varies.

In [ ]:
# YOUR CODE HERE
# ---------------
# For each threshold in [0.6, 0.75, 0.9]:
#   1. Get stable features: indices where max_probs >= threshold
#   2. Compute TP = |stable & TRUE_FEATURES|, FP = |stable - TRUE_FEATURES|
#   3. Compute precision and recall
#   4. Print results

thresholds = [0.6, 0.75, 0.9]

In [ ]:
# AUTO-GRADED TESTS — do not modify
def test_exercise_62():
    thresholds = [0.6, 0.75, 0.9]
    results = {}
    for thr in thresholds:
        selected = set(np.where(max_probs >= thr)[0].tolist())
        tp = len(selected & set(TRUE_FEATURES))
        fp = len(selected - set(TRUE_FEATURES))
        prec = tp / max(len(selected), 1)
        recall = tp / len(TRUE_FEATURES)
        results[thr] = (len(selected), tp, fp, prec, recall)
        print(f"π_thr={thr}: selected={len(selected)}, TP={tp}, FP={fp}, prec={prec:.2f}, recall={recall:.2f}")
    
    # Monotonicity checks: higher threshold → fewer or equal selected
    assert results[0.6][0] >= results[0.75][0] >= results[0.9][0], \
        "Higher threshold should select fewer or equal features"
    print("Test passed: monotonicity holds.")

test_exercise_62()

## Summary

### Key Takeaways

1. **Stability selection converts binary selection into a probability** by repeating the Lasso path on random subsamples. This naturally handles correlated features: all members of a correlated group receive intermediate probabilities rather than one being arbitrarily chosen.

2. **The Meinshausen-Bühlmann bound** provides a finite-sample guarantee on the expected number of false positives. The guarantee requires controlling the average model size (via $\lambda$) and choosing the threshold accordingly.

3. **Single LassoCV is fragile**: running on 30 different subsamples produces very different feature sets, especially for correlated features. Stability selection averages this variability.

4. **Group Lasso produces all-or-nothing group selection** via block soft-thresholding. When feature groups are semantically meaningful, Group Lasso is more interpretable than feature-by-feature selection.

5. **Threshold $\pi_\text{thr}$ controls the precision-recall tradeoff**: higher threshold → higher precision (fewer false positives), lower recall (more true features missed).

### What's Next

Notebook 03 compares tree-based importance methods and the knockoff filter — a method that provides exact FDR control without requiring the subsampling loop.

### Additional Resources
- Meinshausen & Bühlmann (2010) "Stability Selection" — original paper with full proofs
- `sklearn.linear_model.lasso_path` — LARS implementation
- `group_lasso` package on PyPI — production Group Lasso with FISTA solver